In [1]:
import datasets
import pandas as pd
import re
import numpy as np

In [2]:
ds = datasets.load_dataset("theelderemo/genius-lyrics-cleaned", cache_dir="./hf_cache")
df = ds["train"].to_pandas()

In [3]:
artists = ["Gojira", "Tool", "Opeth", "Mastodon", "Death"]
subset = df[df["artist"].isin(artists)].copy()
subset["artist"].value_counts()

artist
Mastodon    154
Opeth       118
Death       116
Gojira       86
Tool         74
Name: count, dtype: int64

In [4]:
subset["clean"] = subset["lyrics"].apply(lambda x: re.sub(r"\[.*?\]", "", x).strip())
subset = subset[subset["clean"].str.len() > 100]
subset["artist"].value_counts()

artist
Mastodon    154
Opeth       118
Death       116
Gojira       86
Tool         74
Name: count, dtype: int64

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X = tfidf.fit_transform(subset["clean"])
y = subset["artist"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv,
scoring="accuracy")
print(f"5-fold CV accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

5-fold CV accuracy: 0.735 (+/- 0.027)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
print(classification_report(y_test, y_pred=clf.predict(X_test)))

              precision    recall  f1-score   support

       Death       0.88      0.83      0.86        18
      Gojira       1.00      0.38      0.56        13
    Mastodon       0.54      0.96      0.69        23
       Opeth       0.93      0.78      0.85        18
        Tool       1.00      0.45      0.62        11

    accuracy                           0.73        83
   macro avg       0.87      0.68      0.71        83
weighted avg       0.83      0.73      0.73        83



In [7]:
from sklearn.model_selection import train_test_split
import os

os.makedirs("data", exist_ok=True)

train_df, eval_df = train_test_split(
    subset, 
    test_size=0.1, 
    stratify=subset["artist"],
    random_state=42
)

print(f"Train: {len(train_df)}, Eval: {len(eval_df)}")
print(train_df["artist"].value_counts())
print(eval_df["artist"].value_counts())

train_df.to_csv("data/train.csv", index=False)
eval_df.to_csv("data/eval.csv", index=False)

Train: 493, Eval: 55
artist
Mastodon    139
Opeth       106
Death       104
Gojira       77
Tool         67
Name: count, dtype: int64
artist
Mastodon    15
Opeth       12
Death       12
Gojira       9
Tool         7
Name: count, dtype: int64
